In [0]:
# Read the landed PostgreSQL baseline without changing any data.

BASELINE_PATH = (
    "abfss://landing@stadblhdeveus2001.dfs.core.windows.net/"
    "postgres/shipments/delivery_id=snapshot_baseline/"
)

spark.conf.set("spark.sql.session.timeZone", "UTC")

baseline_df = spark.read.parquet(BASELINE_PATH)

print("Source schema:")
baseline_df.printSchema()

print("Sample shipment records:")
display(
    baseline_df
    .orderBy("shipment_id")
    .limit(10)
)

print(f"Baseline rows: {baseline_df.count()}")

In [0]:
from uuid import uuid4

from pyspark.sql import functions as F


BASELINE_PATH = (
    "abfss://landing@stadblhdeveus2001.dfs.core.windows.net/"
    "postgres/shipments/delivery_id=snapshot_baseline/"
)

BRONZE_TABLE = (
    "cat_adblh_dev.bronze.shipments_postgres_raw"
)

DELIVERY_ID = "snapshot_baseline"
RUN_ID = uuid4().hex

spark.conf.set("spark.sql.session.timeZone", "UTC")

# Stop before writing if this initial-load target already exists.
if spark.catalog.tableExists(BRONZE_TABLE):
    raise RuntimeError(
        f"Initial load stopped: {BRONZE_TABLE} already exists. "
        "No records were appended or overwritten."
    )

# Capture file lineage directly from the source read.
bronze_candidate_df = (
    spark.read.parquet(BASELINE_PATH)
    .select(
        "shipment_id",
        "order_id",
        "shipment_status",
        "updated_at",
        "source_version",
        F.col("_metadata.file_path").alias("_source_file"),
    )
    .withColumn("_source_system", F.lit("postgres"))
    .withColumn("_delivery_id", F.lit(DELIVERY_ID))
    .withColumn("_run_id", F.lit(RUN_ID))
    .withColumn("_ingested_at", F.current_timestamp())
)

# Create a managed Delta table.
# errorifexists also protects the write if the target appears after the earlier existence check.
# Make the prepared DataFrame available to SQL in this session.
bronze_candidate_df.createOrReplaceTempView(
    "shipment_baseline_candidate"
)

# Intentionally omit IF NOT EXISTS and OR REPLACE.
# An existing target must cause an error.
spark.sql(f"""
    CREATE TABLE {BRONZE_TABLE}
    USING DELTA
    AS SELECT *
    FROM shipment_baseline_candidate
""")

# Read the committed table, rather than displaying the candidate.
persisted_df = spark.table(BRONZE_TABLE)

print(f"Created table: {BRONZE_TABLE}")
print(f"Delivery ID: {DELIVERY_ID}")
print(f"Run ID: {RUN_ID}")

display(
    persisted_df
    .groupBy(
        "_source_system",
        "_delivery_id",
        "_run_id",
        "_source_file",
    )
    .count()
)

display(
    persisted_df
    .orderBy("shipment_id")
    .limit(10)
)

source_rows = spark.read.parquet(BASELINE_PATH).count()
bronze_rows = persisted_df.count()

print(f"Source rows: {source_rows}")
print(f"Persisted Bronze rows: {bronze_rows}")

if source_rows != bronze_rows:
    raise RuntimeError(
        "Post-write row-count check failed. "
        "The table was committed; inspect it before retrying."
    )

print("Initial Bronze row-count reconciliation: PASS")

In [0]:
from functools import reduce

from pyspark.sql import functions as F


BASELINE_PATH = (
    "abfss://landing@stadblhdeveus2001.dfs.core.windows.net/"
    "postgres/shipments/delivery_id=snapshot_baseline/"
)

BRONZE_TABLE = "cat_adblh_dev.bronze.shipments_postgres_raw"

SOURCE_COLUMNS = [
    "shipment_id",
    "order_id",
    "shipment_status",
    "updated_at",
    "source_version",
]

STRING_METADATA = [
    "_source_system",
    "_source_file",
    "_delivery_id",
    "_run_id",
]

source_with_file_df = (
    spark.read.parquet(BASELINE_PATH)
    .select(
        *SOURCE_COLUMNS,
        F.col("_metadata.file_path").alias("_source_file"),
    )
)

source_df = source_with_file_df.select(*SOURCE_COLUMNS)
bronze_df = spark.table(BRONZE_TABLE)
bronze_values_df = bronze_df.select(*SOURCE_COLUMNS)

# Compare source-field names and types before comparing values.
source_types = [
    (field.name, field.dataType.simpleString())
    for field in source_df.schema.fields
]

bronze_types = [
    (field.name, field.dataType.simpleString())
    for field in bronze_values_df.schema.fields
]

if source_types != bronze_types:
    raise RuntimeError(
        "Source-field types differ between landing and Bronze."
    )

# Compare record occurrences in both directions.
missing_rows = source_df.exceptAll(bronze_values_df).count()
extra_rows = bronze_values_df.exceptAll(source_df).count()

# Check required metadata for null or blank values.
invalid_string_conditions = [
    F.col(column).isNull()
    | (F.length(F.trim(F.col(column))) == 0)
    for column in STRING_METADATA
]

invalid_metadata = (
    reduce(lambda left, right: left | right,
           invalid_string_conditions)
    | F.col("_ingested_at").isNull()
)

invalid_metadata_rows = bronze_df.filter(
    invalid_metadata
).count()

wrong_identity_rows = bronze_df.filter(
    (~F.col("_source_system").eqNullSafe(F.lit("postgres")))
    | (
        ~F.col("_delivery_id").eqNullSafe(
            F.lit("snapshot_baseline")
        )
    )
).count()

# Compare source records together with their recorded file lineage.
lineage_columns = SOURCE_COLUMNS + ["_source_file"]

missing_lineage_rows = (
    source_with_file_df.select(*lineage_columns)
    .exceptAll(bronze_df.select(*lineage_columns))
    .count()
)

extra_lineage_rows = (
    bronze_df.select(*lineage_columns)
    .exceptAll(source_with_file_df.select(*lineage_columns))
    .count()
)

checks = {
    "Missing source record occurrences": missing_rows,
    "Extra Bronze record occurrences": extra_rows,
    "Rows with missing or blank metadata": invalid_metadata_rows,
    "Rows with unexpected source or delivery": wrong_identity_rows,
    "Missing record/file lineage occurrences": missing_lineage_rows,
    "Extra record/file lineage occurrences": extra_lineage_rows,
}

for name, failures in checks.items():
    print(f"{name}: {failures}")

if any(failures != 0 for failures in checks.values()):
    raise RuntimeError(
        "Baseline validation failed. No data was changed."
    )

print("Baseline values and lineage validation: PASS")